# OK Agent Python SDK — Jupyter Notebook 示例

本 Notebook 演示如何通过 Python SDK 连接到 OK Agent，进行数据分析、文件操作和交互式查询。

## 前提
1. OK Agent 正在运行: `ok serve`
2. 已安装 Python SDK: `pip install ok-sdk`

In [ ]:
# 安装 SDK（如果尚未安装）
# !pip install ok-sdk

from ok import Agent

agent = Agent()
if not agent.connect():
    print("❌ 请先启动 OK Agent:  ok serve")
else:
    print("✅ 已连接到 OK Agent")

## 示例 1: 简单问答

In [ ]:
result = agent.run("What is the capital of France? Explain briefly.")
print(result.text)
print(f"\nTokens used: {result.usage['total_tokens'] if result.usage else 'N/A'}")

## 示例 2: 数据分析

In [ ]:
result = agent.run("""
Create a small CSV dataset with 10 rows of sample sales data
(date, product, amount, region), then analyze it and tell me:
1. Total sales by region
2. Best-selling product
3. Any trends you notice
""", auto_approve=True)
print(result.text)

## 示例 3: 流式输出（更适合长时间运行的任务）

In [ ]:
agent.submit("Explain the concept of recursion with a Python example.")
for event in agent.stream(auto_approve=True):
    if event.kind == event.TEXT:
        print(event.text, end='', flush=True)
    elif event.kind == event.DONE:
        print("\n\n✅ Done")
        break
    elif event.kind == event.USAGE and event.usage:
        print(f"\n\n[Tokens: {event.usage.total_tokens}, Cost: ${event.usage.cost_usd:.4f}]")

## 示例 4: 文件操作

In [ ]:
result = agent.run("List all Python files in the current directory and count the lines", auto_approve=True)
print(result.text)

## 示例 5: 代码审查（结合 OK 的内置能力）

In [ ]:
result = agent.run(
    "Run a code review on any Go file in the internal/ directory. Check for error handling issues.",
    auto_approve=True
)
print(result.text)

## 示例 6: 多轮对话

In [ ]:
# 第一轮
r1 = agent.run("My name is Alice.")
print("Agent:", r1.text[:100] if r1.text else "(no text)")

# 第二轮 — Agent 应该记得名字
r2 = agent.run("What's my name?")
print("Agent:", r2.text)

## 示例 7: 会话管理

In [ ]:
ctx = agent.get_context()
print(f"Context usage: {ctx['used']} / {ctx['window']} tokens")

history = agent.get_history()
print(f"Conversation turns: {len(history)}")
for msg in history[-3:]:
    print(f"  {msg['role']}: {msg['content'][:60]}...")

## 示例 8: WebSocket 传输（更快）

In [ ]:
# 需要安装 websocket-client:  pip install ok-sdk[ws]
from ok.transport_ws import WebSocketTransport, WSConfig

ws_transport = WebSocketTransport(WSConfig(url="ws://127.0.0.1:3030/ws"))
ws_agent = Agent(transport=ws_transport)

if ws_agent.connect():
    result = ws_agent.run("Say hello via WebSocket!")
    print("WebSocket:", result.text)
    ws_agent.close()
else:
    print("WebSocket 连接失败，请确认 Agent 运行中且支持 /ws")

## 清理

In [ ]:
agent.close()
print("✅ 连接已关闭")